<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/colab/custom_cnn_refactored_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Custom CNN - Version Refactorisée v2

Ce notebook utilise les **5 modules refactorisés** de `src.notebooks` pour un code ultra-propre et modulaire.

## 📦 Nouveaux modules

- `data_utils` - Chargement et preprocessing
- `model_builders` - Construction de modèles
- `training_utils` - Entraînement et évaluation
- `visualization_utils` - Graphiques
- `interpretability_utils` - Grad-CAM

## 1. Configuration et Imports

In [6]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")

    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)

    os.chdir('/content/Data_Pipeline')

    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)

    # Installation du package en mode éditable (sans dépendances - détection Colab dans setup.py)
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")

    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])

    # Extraction models
    archive_models = '/content/drive/MyDrive/DS_COVID/inceptionv3_best.zip'
    if os.path.exists(archive_models):
        print("📦 Extraction models...")
        os.makedirs('./models/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_models, '-d', './models/'])

    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    project_root = Path.cwd().parent.parent

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS DES TRANSFORMERS
# =============================================================================

try:
    from src.features.Pipelines.Transformateurs.image_loaders import ImageLoader
    from src.features.Pipelines.Transformateurs.image_preprocessing import (
        ImageResizer, ImageNormalizer, ImageFlattener, ImageMasker
    )
    from src.features.Pipelines.Transformateurs.image_augmentation import (
        ImageAugmenter, ImageRandomCropper
    )
    from src.features.Pipelines.Transformateurs.image_features import (
        ImageHistogram, ImagePCA, ImageStandardScaler
    )
    print("✅ Transformers importés")
except ImportError as e:
    print(f"⚠️ Erreur import transformers: {e}")


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🎛️ Images: {img_size}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)
print("\n💡 Variables disponibles:")
print("   • config: Configuration complète (Config object)")
print("   • ENV: Environnement actuel")
print("\n🎯 Transformers disponibles:")
print("   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener")
print("   • ImageAugmenter, ImageRandomCropper")
print("   • ImageHistogram, ImagePCA, ImageStandardScaler")
print("=" * 70)
# Fin de la cellule de configuration standalone

🌍 Environnement: WSL
✅ Transformers importés

✅ CONFIGURATION PRÊTE - Data Pipeline
📂 Projet: /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline
📊 Dataset: /home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline/data/raw/COVID-19_Radiography_Dataset/COVID-19_Radiography_Dataset
🏷️ Classes: COVID, Normal, Lung_Opacity, Viral Pneumonia
🎛️ Images: (256, 256)
🔧 Batch: 32 | Époques: 50
📐 Dataset accessible: ✅

💡 Variables disponibles:
   • config: Configuration complète (Config object)
   • ENV: Environnement actuel

🎯 Transformers disponibles:
   • ImageLoader, ImageResizer, ImageNormalizer, ImageFlattener
   • ImageAugmenter, ImageRandomCropper
   • ImageHistogram, ImagePCA, ImageStandardScaler


In [7]:
# Imports des fonctions utilitaires depuis les 5 nouveaux modules
from src.notebooks import (
    # Data utils
    load_dataset,
    create_preprocessing_pipeline,
    prepare_train_val_test_split,
    compute_class_weights,
    create_data_generators,
    
    # Model builders
    build_custom_cnn,
    compile_model,
    create_callbacks,
    
    # Training utils
    train_model,
    evaluate_model,
    
    # Visualization utils
    plot_training_curves,
    plot_confusion_matrix,
    
    # Interpretability utils
    setup_interpretability,
    select_sample_images,
    run_gradcam_analysis,
)

print("✅ Fonctions utilitaires importées depuis les 5 modules refactorisés")
print("\n📦 Modules utilisés:")
print("   • data_utils")
print("   • model_builders")
print("   • training_utils")
print("   • visualization_utils")
print("   • interpretability_utils")

✅ Fonctions utilitaires importées depuis les 5 modules refactorisés

📦 Modules utilisés:
   • data_utils
   • model_builders
   • training_utils
   • visualization_utils
   • interpretability_utils


In [8]:
# Définition des paramètres principaux
VERBOSE = True
LOAD_MASKS = False  # True pour version maskée
N_IMAGES_PER_CLASS = None  # None = Charger toutes les images
AUGMENT_TRAIN = True  # True pour augmenter les données d'entraînement
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE_EARLY_STOP = 15
PATIENCE_REDUCE_LR = 5
RANDOM_SEED = 42

## 2. Chargement et Préparation des Données

In [9]:
# ⚠️ NOUVEAU: load_dataset retourne 4 valeurs au lieu de 2
# (image_paths, mask_paths, labels, labels_int)
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=N_IMAGES_PER_CLASS,  # None = Charger toutes les images
    load_masks=LOAD_MASKS,  # True pour version maskée
    verbose=VERBOSE
)

CHARGEMENT DES DONNÉES
  COVID               : 3616 images
  Normal              : 10192 images
  Lung_Opacity        : 6012 images
  Viral Pneumonia     : 1345 images

  Total: 21165 images
  Classes: 4
  Distribution: [ 3616 10192  6012  1345]
  Lung_Opacity        : 6012 images
  Viral Pneumonia     : 1345 images

  Total: 21165 images
  Classes: 4
  Distribution: [ 3616 10192  6012  1345]


In [10]:
# Créer la pipeline de preprocessing
pipeline = create_preprocessing_pipeline(
    img_size=(128, 128),
    color_mode='RGB',
    mask_paths=None, # Pas de masques pour l'instant
    verbose=VERBOSE
)

# Charger et préprocesser les images
print("\n🔄 Chargement des images...")
images = pipeline.fit_transform(image_paths)
images = images.astype('float32') / 255.0

print(f"\n📊 Images préparées:")
print(f"   Shape: {images.shape}")
print(f"   Range: [{images.min():.3f}, {images.max():.3f}]")
print(f"   Dtype: {images.dtype}")

PREPROCESSING PIPELINE

✅ Pipeline créée avec 2 étapes

🔄 Chargement des images...


Loading images:  37%|███▋      | 7740/21165 [00:23<00:41, 322.57it/s]



KeyboardInterrupt: 

In [ ]:
# Split train/val/test
X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat = prepare_train_val_test_split(
    images=images,
    labels_int=labels_int,
    num_classes=len(config.classes),
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    random_seed=RANDOM_SEED,
    verbose=VERBOSE
)

# Récupérer les labels integer pour le calcul des class weights et confusion matrix
y_train = np.argmax(y_train_cat, axis=1)
y_val = np.argmax(y_val_cat, axis=1)
y_test = np.argmax(y_test_cat, axis=1)

In [ ]:
# Calculer les class weights
class_weights = compute_class_weights(
    y_train=y_train,
    categories=config.classes,
    verbose=VERBOSE
)

In [ ]:
# Créer les data generators
train_generator, val_generator, test_generator = create_data_generators(
    X_train=X_train,
    y_train_cat=y_train_cat,
    X_val=X_val,
    y_val_cat=y_val_cat,
    X_test=X_test,
    y_test_cat=y_test_cat,
    batch_size=BATCH_SIZE,
    augment_train=AUGMENT_TRAIN,
    verbose=VERBOSE
)

## 3. Construction et Compilation du Modèle

In [ ]:
# Construire le modèle Custom CNN
model = build_custom_cnn(
    input_shape=(128, 128, 3),
    num_classes=len(config.classes),
    verbose=VERBOSE
)

# Afficher le résumé
model.summary()

In [ ]:
# Compiler le modèle
model = compile_model(
    model=model,
    learning_rate=LEARNING_RATE,
    verbose=VERBOSE
)

In [ ]:
# Créer les callbacks
callbacks = create_callbacks(
    models_dir=config.results_dir / 'custom_cnn_models',
    monitor='val_accuracy',
    patience_early_stop=PATIENCE_EARLY_STOP,
    patience_reduce_lr=PATIENCE_REDUCE_LR,
    verbose=VERBOSE
)

## 4. Entraînement

In [ ]:
# Entraîner le modèle
history = train_model(
    model=model,
    train_generator=train_generator,
    val_generator=val_generator,
    class_weights=class_weights,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=VERBOSE  # Progress bar
)

## 5. Visualisation des Courbes d'Apprentissage

In [ ]:
# Plot training curves
plots_dir = config.results_dir / 'custom_cnn_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

plot_training_curves(
    history=history,
    figsize=(16, 5),
    save_path=plots_dir / 'training_curves.png'
)
plt.show()

## 6. Évaluation

In [ ]:
# ⚠️ NOUVEAU: evaluate_model accepte maintenant un tuple (X_test, y_test_cat)
# au lieu d'un generator (plus sûr pour garantir l'ordre)
results = evaluate_model(
    model=model,
    test_data=(X_test, y_test_cat),  # Tuple au lieu de generator
    class_names=config.classes,
    verbose=VERBOSE
)

# Extraire les prédictions
y_pred = results['y_pred']
y_pred_probs = results['y_pred_probs']

In [ ]:
# Matrice de confusion
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    normalize=True,
    figsize=(10, 8),
    save_path=plots_dir / 'confusion_matrix.png'
)
plt.show()

## 7. Interprétabilité - Grad-CAM

In [ ]:
# Setup Grad-CAM
gradcam = setup_interpretability(
    model=model,
    verbose=VERBOSE
)

In [ ]:
# Sélectionner des images échantillons (correctement classifiées)
indices_correct, descriptions_correct = select_sample_images(
    X_data=X_test,
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    n_samples=2,  # 2 par classe
    strategy='correct',
    random_state=RANDOM_SEED
)

print(f"\n📊 {len(indices_correct)} échantillons correctement classifiés sélectionnés")

In [ ]:
# Analyse Grad-CAM sur les échantillons correctement classifiés
interp_dir = config.results_dir / 'interpretability_custom_cnn'
interp_dir.mkdir(parents=True, exist_ok=True)

run_gradcam_analysis(
    gradcam=gradcam,
    X_data=X_test,
    indices=indices_correct,
    descriptions=descriptions_correct,
    class_names=config.classes,
    y_pred_probs=y_pred_probs,
    figsize=(15, 4),
    save_dir=interp_dir
)

In [ ]:
# Sélectionner aussi quelques échantillons mal classifiés
indices_incorrect, descriptions_incorrect = select_sample_images(
    X_data=X_test,
    y_true=y_test,
    y_pred=y_pred,
    class_names=config.classes,
    n_samples=1,  # 1 par classe si disponible
    strategy='incorrect',
    random_state=RANDOM_SEED
)

if len(indices_incorrect) > 0:
    print(f"\n📊 {len(indices_incorrect)} échantillons mal classifiés trouvés")
    
    # Analyse Grad-CAM sur les erreurs
    run_gradcam_analysis(
        gradcam=gradcam,
        X_data=X_test,
        indices=indices_incorrect,
        descriptions=descriptions_incorrect,
        class_names=config.classes,
        y_pred_probs=y_pred_probs,
        figsize=(15, 4),
        save_dir=interp_dir / 'errors'
    )
else:
    print("\n✅ Aucune erreur trouvée (modèle parfait sur le test set!)")

## 8. Résumé

✅ Notebook refactorisé avec les **5 nouveaux modules** !

### 🎯 Changements par rapport à v1

1. **Imports modulaires** : 5 modules spécialisés au lieu d'un seul
2. **load_dataset()** : Retourne 4 valeurs (image_paths, mask_paths, labels, labels_int)
3. **evaluate_model()** : Accepte tuple (X_test, y_test) au lieu de generator (plus sûr)
4. **setup_interpretability()** : Nouvelle fonction pour initialiser Grad-CAM
5. **select_sample_images()** : Sélection stratégique (correct/incorrect/random/one_per_class)
6. **run_gradcam_analysis()** : Analyse simplifiée avec visualisation automatique

### 📦 Architecture des modules

```
src/notebooks/
├── __init__.py                 # Exports centralisés
├── data_utils.py              # 7 fonctions - Data loading & preprocessing
├── model_builders.py          # 5 fonctions - Custom CNN & Transfer Learning
├── training_utils.py          # 2 fonctions - Training & evaluation
├── visualization_utils.py     # 2 fonctions - Plots
└── interpretability_utils.py  # 3 fonctions - Grad-CAM
```

**Total : 18 fonctions réutilisables**

In [ ]:
print("=" * 70)
print("🎉 NOTEBOOK TERMINÉ")
print("=" * 70)
print("\n✅ Modèle entraîné et évalué")
print("✅ Visualisations générées")
print("✅ Interprétabilité analysée")
print("\n📊 Résultats sauvegardés dans:")
print(f"   • Modèles: {config.results_dir / 'custom_cnn_models'}")
print(f"   • Plots: {plots_dir}")
print(f"   • Grad-CAM: {interp_dir}")